[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/boltzgen/advanced/08_antibody_fab/08_fab_lab.ipynb)


# 08 — 항체 Fab 실습 + developability

> 본문 [`08_antibody_fab.md`](08_antibody_fab.md) 와 **한 절씩 짝지어** 보세요.
> 이 노트북의 표·그래프·수치는 **여러분이 직접 돌린 결과**(`my_run/`)에서 계산합니다.
> **① 직접 설계 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서로 갑니다. 설계 셀은 NVIDIA GPU 전용이에요(CPU 폴백 없음) — Colab 이면 **런타임 → 런타임 유형 변경 → T4 GPU**.

## 0) 준비 — 저장소 클론 & 작업 폴더 이동

이 셀이 저장소를 클론하고 `08_antibody_fab/` 로 이동합니다. 로컬에서 `08_antibody_fab/` 안에 열었다면 클론 없이 진행돼요.

In [ ]:
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # fork 했다면 본인 주소로 바꾸세요
CLONE_AS = "bio-guides"
CHAPTER  = "08_antibody_fab"
PIP_PKGS = "pandas matplotlib gemmi py3Dmol"   # 없으면 설치할 분석 라이브러리

import os, sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules

# HF 가중치 다운로드가 멈춘 채 매달리지 않도록 타임아웃을 겁니다.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

_MARK = "boltzgen_viz.py"          # 이 파일이 있는 폴더가 advanced/ 루트

def _find_root():
    """advanced/ 루트를 찾습니다."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):   # 클론 직후: cwd 아래로만 깊이 3까지
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "advanced/ 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)          # 이 챕터 폴더로 이동 → data/ 상대경로가 그대로 동작
sys.path.insert(0, str(ADV_ROOT))     # boltzgen_viz import 보장
import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")

# import 안 되는 패키지만 설치합니다.
import importlib, importlib.util
_IMPORT_NAME = {"scikit-learn": "sklearn", "pillow": "PIL", "biopython": "Bio"}
def _have(mod):
    try: return importlib.util.find_spec(mod) is not None
    except Exception: return False
def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p))]
    if miss:
        print("필요 라이브러리 설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))  # python -m pip (Ch.03 권고)
        importlib.invalidate_caches()
if PIP_PKGS:
    _ensure(PIP_PKGS)

# 내 결과는 my_run/ 에 쌓이고, 없으면 커밋된 레퍼런스로 폴백합니다.
MY  = pathlib.Path("my_run")
MY.mkdir(exist_ok=True)

def find_one(pattern, ref, quiet=False):
    """my_run/ 에서 먼저 찾고, 없으면 레퍼런스 폴더에서 찾습니다."""
    for base in (MY/"final_ranked_designs", MY/"intermediate_designs_inverse_folded", MY):
        hits = sorted(base.glob(pattern))
        if hits:
            if not quiet: print(f"[내 결과]   {hits[0]}")
            return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"{ref} 에서 '{pattern}' 을 찾지 못했습니다."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def cols_in(df, *names):
    """내 실행 결과와 레퍼런스는 컬럼 구성이 조금 다를 수 있어, 있는 컬럼만 고른다."""
    missing = [c for c in names if c not in df.columns]
    if missing:
        print("(이 실행에는 없는 컬럼 — 건너뜁니다:", ", ".join(missing) + ")")
    return [c for c in names if c in df.columns]

def inherit_run(*rel_paths):
    """앞 챕터에서 돌린 설계 결과를 이어받습니다(없으면 레퍼런스로 폴백)."""
    global MY
    if (MY / "final_ranked_designs").exists():
        return MY
    for rel in rel_paths:
        p = pathlib.Path(rel)
        if (p / "final_ranked_designs").exists():
            print("[이어받기] 앞 챕터에서 직접 돌린 결과를 사용합니다:", p)
            MY = p
            return MY
    return MY

# 레퍼런스 그림을 덮어쓰지 않도록 my_ 접두어.
def my_fig(name):
    return f"my_{name}"

print("작업 폴더 :", pathlib.Path.cwd())

## 1) 직접 돌려보기 — 내 결과 만들기

- 학습용 규모 `num_designs=8 --budget=4` (레퍼런스 결과는 num_designs=30)
- 소요 시간 실측 **585초**(최종 4개) — **24GB GPU · 가중치 캐시** 기준이에요. Colab T4 는 가속 커널이 꺼져 더 걸리고(T4 실측치 없음), 첫 실행은 가중치 ~6GB 다운로드가 더 붙어요.
- 건너뛰면 아래 분석이 커밋된 레퍼런스 결과로 이어집니다.

In [ ]:
SPEC, PROTOCOL = "example/fab_targets/pdl1.yaml", "antibody-anything"
NUM_DESIGNS, BUDGET = 8, 4

import shutil
OUT = MY.resolve()

def _gpu():
    try:
        import torch
        return torch.cuda.is_available()
    except ImportError:
        return shutil.which("nvidia-smi") is not None

if not _gpu():
    print("GPU 런타임이 아니라 설계 실행을 건너뜁니다 — 아래 분석은 레퍼런스 결과로 이어집니다.")
    print("  · 직접 돌리려면 Colab [런타임 → 런타임 유형 변경 → T4 GPU] 후 이 셀을 다시 실행하세요.")
else:
    SRC = ADV_ROOT / ".boltzgen_src"          # 예제 명세·타깃 CIF 가 들어 있는 BoltzGen 소스
    if not SRC.exists():
        _run(f'git clone --depth 1 https://github.com/HannesStark/boltzgen.git "{SRC}"')
    if not _have("boltzgen"):
        _run(f'"{sys.executable}" -m pip -q install -e "{SRC}"')

    # 이 예제의 타깃 구조는 BoltzGen 레포에 커밋돼 있지 않아 먼저 받아옵니다(있으면 건너뜀).
    import urllib.request
    for _rel, _url in [("example/fab_targets/7uxq.cif", "https://files.rcsb.org/download/7uxq.cif")]:
        _dst = SRC / _rel
        if _dst.exists():
            print("타깃 파일 이미 있음:", _rel)
        else:
            _dst.parent.mkdir(parents=True, exist_ok=True)
            print("타깃 파일 내려받는 중:", _rel)
            urllib.request.urlretrieve(_url, _dst)
    try:
        _run(f'cd "{SRC}" && boltzgen run {SPEC} --output "{OUT}" --protocol {PROTOCOL} '
             f'--num_designs {NUM_DESIGNS} --budget {BUDGET}')
        print("\n내 결과 →", OUT / "final_ranked_designs")
    except Exception as e:
        print("\n설계 실행이 도중에 멈췄어요 —", type(e).__name__)
        print("  · 이 셀을 다시 실행하면 같은 --output 산출물을 재사용해 이어서 끝냅니다(실측 763초 → 재개 486초).")
        print("  · GPU 메모리가 부족했다면 NUM_DESIGNS 4, BUDGET 2 로 줄이세요.")

## 2) 무엇이 실제로 돌았나 — `steps.yaml` (본문 8.3)

`antibody-anything` 은 `protein-anything` 과 스텝 구성부터 달라요.

- **inverse folding 에서 Cys 자동 금지** — CDR 에 자유 시스테인이 생기면 도메인 안의 이황화 짝이 어긋나요.
- **design_folding 생략** — CDR 은 framework·타깃에 의존적이라 단독 폴딩이 무의미해요.
- 최대 소수성 패치 미계산.

레퍼런스를 만든 명령은 이거예요. 타깃 `7uxq.cif` 는 BoltzGen 레포에 없어서 먼저 받아야 하고, 위 설계 셀이 그 다운로드까지 해 둡니다.

```bash
curl -sSL -o example/fab_targets/7uxq.cif https://files.rcsb.org/download/7uxq.cif
boltzgen run example/fab_targets/pdl1.yaml --output workbench/fab \
  --protocol antibody-anything --num_designs 30 --budget 10
```

스텝 구성은 외우지 말고, 실행이 출력 폴더에 남긴 `steps.yaml` 에서 그대로 읽어요.

In [ ]:
steps_file = find_one("steps.yaml", "data/fab")
names = [ln.split(":", 1)[1].strip()
         for ln in steps_file.read_text().splitlines()
         if ln.strip().startswith("- name:")]

print("이 출력 폴더에서 실행된 스텝", len(names), "개")
for i, n in enumerate(names, 1):
    print(f"  {i}. {n}")
print()
print("design_folding 포함 —", "예" if "design_folding" in names else "아니오 (antibody-anything 은 이 스텝을 생략)")
print("이 목록은 '이 출력 폴더에서 실제로 돈 스텝'이에요 — 앞 스텝 산출물을 재사용해 이어 돌리면 그만큼만 남습니다.")

## 3) 결과 표 — 무엇을 보나 (본문 8.6)

돌아간 스텝이 남긴 건 메트릭 CSV 한 장이에요. 본문 8.6 이 꼽는 다섯 가지를 한 줄에 모아 봅니다 —
`design_to_target_iptm`(PD-L1 결합), `design_ptm`·`filter_rmsd`(구조·자기일관성),
`num_design`(재설계 CDR 길이), `liability_score`(개발성).

In [ ]:
from boltzgen_viz import load_metrics, metrics_overview
import pandas as pd

CSV = str(find_one("final_designs_metrics_*.csv", "data/fab"))
df = pd.read_csv(CSV).sort_values("final_rank").reset_index(drop=True)
print("디자인", len(df), "개 | 전체 컬럼", df.shape[1], "종")
df[cols_in(df, "final_rank", "id", "design_ptm", "design_to_target_iptm",
           "filter_rmsd", "plip_hbonds_refolded", "num_design", "liability_score")]

## 4) 메트릭 그래프 — 순위와 함께 보기 (본문 8.7)

표의 숫자가 순위를 따라 어떻게 움직이는지는 그림이 빨라요. pTM(보라)·ipTM(주황)·RMSD(청록) 바에
길이–H-bond 산점도가 붙습니다. 레퍼런스 그림(`08_fab_metrics.png`)을 덮지 않도록 `my_` 접두어로 저장돼요.

In [ ]:
rows = load_metrics(CSV)
FIG = my_fig("08_fab_metrics.png")
metrics_overview(rows, "Antibody Fab (PD-L1) — Design Metrics Overview", FIG)
from IPython.display import Image; Image(FIG)

## 5) 재설계된 곳은 CDR 뿐인가 — `num_design` 과 자유 Cys (본문 8.4·8.7)

레퍼런스에서는 ipTM 이 rank 4(0.486)·rank 5(0.482)·rank 1(0.463) 순으로 높고, RMSD 는 rank 1·2·3 이 2Å 미만이었어요.
그런데 이 숫자는 **CDR 만 바뀌었을 때**만 의미가 있어요 — framework 까지 흔들렸다면 scaffold 설계가 아니니까요.

각 scaffold YAML 이 세 요소로 "어디를 건드릴지"를 못 박아요(본문 8.4).

```yaml
# adalimumab.6cr1.yaml (요약)
design:            [ { chain: { id: H, res_index: 27..38,56..65,99..110 } } ]        # CDR 재설계
not_design:        [ { chain: { id: H, res_index: 1..26,39..55,66..98,111.. } } ]    # framework 고정
design_insertions: [ { insertion: { id: H, res_index: 99, num_residues: 1..12 } } ]  # CDR3 길이 가변
```

`num_design` 은 이렇게 열어 준 구간의 총 길이예요(본문 8.7의 48~60aa). Fab 는 사슬이 둘이니
`designed_sequence_1`·`designed_sequence_2` 길이를 각각 세서 합이 `num_design` 과 맞는지, 그리고
그 구간에 **자유 Cys 가 하나도 없는지** 확인합니다.

In [ ]:
des_cols = cols_in(df, "designed_sequence_1", "designed_sequence_2") or cols_in(df, "designed_sequence")
has_nd = "num_design" in df.columns

for _, r in df.head(5).iterrows():
    parts = [str(r[col]) for col in des_cols]
    lens = "+".join(str(len(p)) for p in parts)
    nd = f"num_design={int(r['num_design']):3d} | " if has_nd else ""
    print(f"rank{int(r['final_rank'])} {r['id']:8s} {nd}사슬별 재설계 {lens} = {sum(len(p) for p in parts):3d} "
          f"| 재설계 구간 Cys={sum(p.count('C') for p in parts)}")

print()
if has_nd:
    print("재설계 영역 길이 범위", int(df["num_design"].min()), "~", int(df["num_design"].max()), "aa")
    print("→ 사슬별 합이 num_design 과 같으면 설계가 CDR 구간 밖으로 새지 않은 거예요.")
print("→ 재설계 구간 Cys 가 0 이면 antibody-anything 의 Cys 자동 금지가 지켜진 겁니다 (본문 8.3).")

## 6) Developability — liability (본문 8.5)

CDR 은 규칙대로 만들어졌어요. 다음 질문은 **약이 될 수 있는가**예요. ipTM 이 좋아도 Met 산화·Asp 절단·소수성 패치가
몰려 있으면 발현·정제·보관에서 무너지거든요. 종합 지표 `liability_score` 는 **낮을수록** 개발하기 쉬운 후보고요(본문 8.5).

모티프별 검출 내역은 `liability_*_count` 가 아니라 `liability_violations_summary` 문자열에 들어 있어요
(`ProtTrypx7(pos18-105,sev10); MetOx(pos4,sev5); …` 형태).

In [ ]:
import collections

score_cols = cols_in(df, "final_rank", "id", "liability_score", "liability_num_violations",
                     "liability_high_severity_violations", "liability_medium_severity_violations")
tbl = df[score_cols]
if "liability_score" in tbl.columns:
    tbl = tbl.sort_values("liability_score")
print("liability_score 낮은 순 (낮을수록 개발성 우수)")
print(tbl.to_string(index=False))

if "liability_violations_summary" in df.columns:
    tot = collections.Counter()
    for summary in df["liability_violations_summary"].astype(str):
        for item in summary.split(";"):
            head = item.strip().split("(", 1)[0]
            if not head:
                continue
            name, mult = head, 1
            if "x" in head:
                stem, _, tail = head.rpartition("x")
                if tail.isdigit():
                    name, mult = stem, int(tail)
            tot[name] += mult
    print()
    print("최종셋 전체에서 검출된 위험 모티프 (합계)")
    for name, n in tot.most_common():
        print(f"  {name:12s} {n}")

print()
print("→ 같은 ipTM 대라면 liability_score 가 낮은 쪽이 임상으로 갈 확률이 높아요.")

## 7) Framework 보존 — VH·VL 두 사슬 모두 (본문 8.6)

개발성까지 봤으니 마지막은 "그래프팅이 정상이었나"예요. Fab 는 한 디자인이 **두 사슬**이라 메트릭 CSV 도
`full_sequence_1`·`full_sequence_2` 두 칸에 나눠 담아요. 한 칸만 읽으면 중쇄와 경쇄 중 하나를 통째로 놓칩니다.

- 중쇄(VH) — `EVQLVE…`·`QVQLQE…` 로 시작해 J-region `…WGQGTLVTVSS` 로 끝나요.
- 경쇄 κ — `DIQMTQ…`·`DIVMTQ…` → `…FGQGTKVEIK`
- 경쇄 λ — `ESVLTQ…` → `…FGGGTKLTVL`

그리고 도메인 안의 **보존 Cys 2개**(framework 이황화쌍)는 사슬마다 그대로 남아야 해요.

In [ ]:
import collections

def chain_kind(s):
    """말단 J-region 모티프로 중쇄/경쇄(κ/λ) 판별 — N말단 prefix 보다 견고."""
    if "WGQG" in s[-15:] or "WGRG" in s[-15:]:                # 중쇄 J: …WGQGT(L/T/M)VTVSS
        return "VH"
    if s.endswith(("EIK", "EIKR")) or "FGQGTK" in s[-14:]:    # κ 경쇄 J: …FGQGTKVEIK
        return "VL-k"
    if "LTVL" in s[-7:] or "FGGGTK" in s[-14:] or "FGSGTK" in s[-14:]:   # λ 경쇄 J: …FGGGTKLTVL
        return "VL-l"
    return "?"

chain_cols = cols_in(df, "full_sequence_1", "full_sequence_2") or cols_in(df, "designed_chain_sequence")

for _, r in df.head(5).iterrows():
    for col in chain_cols:
        s = str(r[col])
        print(f"rank{int(r['final_rank'])} {r['id']:8s} {chain_kind(s):5s} len={len(s):4d} "
              f"Cys={s.count('C')} | {s[:6]}… …{s[-11:]}")

kinds = collections.Counter(chain_kind(str(r[col])) for _, r in df.iterrows() for col in chain_cols)
print()
print("사슬 종류 분포", dict(kinds))
print("→ 한 디자인에서 VH 와 VL 이 하나씩 잡히고 사슬당 Cys 가 2개면 CDR 만 바뀐 정상 그래프팅이에요.")
print("→ framework 가 많이 변했으면 scaffold YAML 의 not_design 범위를 넓혀 더 고정하세요 (본문 8.6).")

## 8) 어떤 후보를 실험으로 보낼까 (본문 8.7)

결합(ipTM)·구조(RMSD)·개발성(liability_score) 세 축은 서로 다른 후보를 가리켜요.
세 축 상위 3개를 각각 뽑아 **교집합에 남는 후보**가 균형 잡힌 후보예요.

In [ ]:
axes = [("결합  ipTM 높음", "design_to_target_iptm", False),
        ("구조  RMSD 낮음", "filter_rmsd", True),
        ("개발성 liability_score 낮음", "liability_score", True)]

picks = []
for label, col, asc in axes:
    if col not in df.columns:
        continue
    top3 = df.sort_values(col, ascending=asc).head(3)
    picks.append(set(top3["id"]))
    shown = ", ".join(f"{i}({v:.3f})" if isinstance(v, float) else f"{i}({v})"
                      for i, v in zip(top3["id"], top3[col]))
    print(f"{label:26s} {shown}")

both = set.intersection(*picks) if picks else set()
print()
if both:
    print("세 축을 모두 만족하는 후보 —", ", ".join(sorted(both)))
    print("→ 이 후보부터 발현·SPR 로 검증하세요.")
else:
    print("세 축을 모두 만족하는 후보 없음 — 표본이 작아 세 조건이 한 디자인으로 모이지 않았어요.")
    print("→ num_designs 를 키워 꼬리를 더 뽑아야 합니다.")

## 9) 만든 Fab 을 눈으로 — rank 1 디자인 3D (본문 8.4·8.6)

5)절에서 "재설계는 CDR 안에 갇혀 있다"를 길이로 확인했어요. 같은 것을 구조에서 봅니다.

**주황·빨강이 설계한 VH·VL 두 사슬, 파랑이 타깃 PD-L1** 이고,
**분홍 stick 이 재설계된 CDR 구간**이에요. CDR 잔기 번호는 적어 두지 않고
결과 CSV 의 `designed_sequence_*` 를 `full_sequence_*` 위에 되짚어 찾습니다. 마우스로 회전·확대할 수 있어요(휠 = 줌, 드래그 = 회전). 구조가 안 보이면 `py3Dmol` 설치 로그를 확인하고 셀을 한 번 더 실행하세요.

In [ ]:
import importlib.util, sys, pathlib
for _pkg in ("py3Dmol", "gemmi"):                       # 없으면 그 자리에서 설치
    if importlib.util.find_spec(_pkg) is None:
        _run(f'"{sys.executable}" -m pip -q install {_pkg}')
import py3Dmol, gemmi

C_DESIGN  = ["#e8883a", "#c0392b"]              # 설계 사슬 — 주황·빨강
C_TARGET  = ["#3477b5", "#7f8c8d", "#8e44ad"]   # 타깃 단백질 — 파랑·회색·보라
C_NUCLEIC = "#1aa090"                           # 타깃 핵산 — 청록

def chain_seq3d(ch):
    """사슬의 한 글자 서열(폴리머 잔기만)."""
    poly = ch.get_polymer()
    return gemmi.one_letter_code([r.name for r in poly]).upper() if len(poly) else ""

def chain_kind3d(ch):
    """protein / dna / rna / hetero(리간드·금속) 판별."""
    poly = ch.get_polymer()
    if not len(poly):
        return "hetero"
    t = str(poly.check_polymer_type())
    return "dna" if "Dna" in t else "rna" if "Rna" in t else "protein"

def load_design(pattern, ref, row=None,
                seq_cols=("full_sequence_1", "full_sequence_2", "designed_chain_sequence")):
    """최종 CIF 를 찾아 열고 (pdb문자열, model, 설계사슬, 타깃사슬, 사슬→CSV컬럼) 을 돌려줍니다.
    설계 사슬은 CSV 의 설계 서열과 사슬 서열이 같은 것으로 정해요 — 사슬 라벨을 하드코딩하지 않습니다."""
    cif = find_one(pattern, ref)                        # [내 결과]/[레퍼런스] 를 스스로 출력합니다
    print("띄울 구조 :", cif, "→", "내 결과 my_run/" if "my_run" in str(cif) else "커밋된 레퍼런스 data/")
    st = gemmi.read_structure(str(cif)); st.setup_entities()
    model = st[0]

    want = {}
    if row is not None:
        for col in seq_cols:
            v = row[col] if col in getattr(row, "index", []) else None
            if isinstance(v, str) and v.strip():
                want.setdefault(v.strip().upper(), col)   # 같은 서열이면 먼저 온 컬럼을 씁니다
    match  = {ch.name: want[chain_seq3d(ch)] for ch in model if chain_seq3d(ch) in want}
    design = list(match)
    if not design:                                        # 폴백 — 설계물은 보통 가장 짧은 단백질 사슬
        prot = [ch for ch in model if chain_kind3d(ch) == "protein"]
        design = [min(prot, key=lambda c: len(c.get_polymer())).name] if prot else []
        print("  (CSV 서열과 일치하는 사슬이 없어 가장 짧은 단백질 사슬을 설계물로 봅니다)")
    target = [ch.name for ch in model if ch.name not in design]

    for ch in model:
        n = len(ch.get_polymer()) or len(ch)
        print(f"  사슬 {ch.name:<2s} {'설계' if ch.name in design else '타깃'} "
              f"| {chain_kind3d(ch):<7s} {n:>4d} | {chain_seq3d(ch)[:28]}")
    return st.make_pdb_string(), model, design, target, match

def complex_view(pdb, model, design, target, width=760, height=540):
    """설계 사슬(주황)·타깃 단백질(파랑)·핵산(청록) cartoon + 비폴리머(리간드 stick·이온 sphere)."""
    view = py3Dmol.view(width=width, height=height)
    view.addModel(pdb, "pdb")
    view.setStyle({}, {"cartoon": {"color": "#cfd6dd"}})
    ti = 0
    for ch in model:
        kind = chain_kind3d(ch)
        if kind == "hetero":
            continue
        if ch.name in design:
            color = C_DESIGN[design.index(ch.name) % len(C_DESIGN)]
        elif kind in ("dna", "rna"):
            color = C_NUCLEIC
        else:
            color = C_TARGET[ti % len(C_TARGET)]; ti += 1
        style = {"cartoon": {"color": color}}
        if kind in ("dna", "rna"):
            style["stick"] = {"color": color, "radius": 0.12}   # 염기까지 보이게
        view.setStyle({"chain": ch.name}, style)
    for name, natoms in {r.name: len(r) for ch in model for r in ch if r.het_flag == "H"}.items():
        if natoms == 1:                                   # 금속·이온
            view.addStyle({"resn": name}, {"sphere": {"color": "#6c5ce7", "radius": 0.9}})
        else:                                             # 리간드
            view.addStyle({"resn": name}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.22}})
    view.setBackgroundColor("white")
    return view

def designed_segments(full, des, min_len=3):
    """이어붙여 저장된 재설계 구간(des)을 전체 서열(full) 위에 되짚어 (시작,끝) 1-based 목록으로.
    CDR 번호를 노트북에 적어 두지 않고 결과 CSV 에서 되찾기 위한 것입니다."""
    full, des = str(full).upper(), str(des).upper()
    out, i, pos = [], 0, 0
    while i < len(des):
        best, at = 0, -1
        for L in range(len(des) - i, min_len - 1, -1):
            j = full.find(des[i:i + L], pos)
            if j >= 0:
                best, at = L, j
                break
        if best == 0:
            i += 1
            continue
        out.append((at + 1, at + best))
        i += best; pos = at + best
    return out

def seg_resi(model, chain_id, segs):
    """서열 위 구간 → 그 사슬의 실제 잔기 번호 목록(3Dmol 의 resi 선택용)."""
    ch = {c.name: c for c in model}[chain_id]
    nums = [r.seqid.num for r in ch.get_polymer()]
    return [n for a, b in segs for n in nums[a - 1:b]]

# 사슬을 찾아준 컬럼 → 그 사슬의 '재설계 구간' 컬럼
DES_COL = {"full_sequence_1": "designed_sequence_1", "full_sequence_2": "designed_sequence_2",
           "designed_chain_sequence": "designed_sequence"}

top = df.iloc[0]                                  # df 는 이미 final_rank 순으로 정렬돼 있어요
print("rank 1 디자인 :", top["id"])
pdb, model, DESIGN, TARGET, MATCH = load_design(
    f"final*designs/*{top['id']}*.cif", "data/fab", row=top)

view = complex_view(pdb, model, DESIGN, TARGET)
for cid in DESIGN:
    full_col = MATCH.get(cid)
    des_col  = DES_COL.get(full_col)
    if not (des_col and des_col in top.index):
        print(f"  사슬 {cid} — 재설계 구간 컬럼이 없어 강조를 건너뜁니다")
        continue
    segs = designed_segments(top[full_col], top[des_col])
    resi = seg_resi(model, cid, segs)
    view.addStyle({"chain": cid, "resi": resi}, {"cartoon": {"color": "#e2247f"}})
    view.addStyle({"chain": cid, "resi": resi},
                  {"stick": {"colorscheme": "magentaCarbon", "radius": 0.18}})
    print(f"  사슬 {cid} ({des_col}) — CDR {len(segs)}구간 {segs} = {len(resi)}잔기 (분홍)")

view.zoomTo()
view.show()

print("\n무엇을 봐야 하나")
print("  · 분홍 CDR 이 타깃(파랑) 쪽을 향해 모여 있으면 정상이에요 — 항체 반대쪽 끝에 흩어져 있으면 결합 포즈가 틀린 겁니다.")
print("  · 주황·빨강 두 사슬이 β-sandwich 두 덩어리로 마주 보고 붙어 있으면 VH-VL 짝이 살아 있는 거예요.")
print("  · framework(회색빛 cartoon) 가 풀려 있으면 5)·7)절의 길이·Cys 검증을 다시 보세요.")
if {"design_to_target_iptm", "filter_rmsd"} <= set(df.columns):
    print(f"\n이 디자인의 숫자 — ipTM {top['design_to_target_iptm']:.3f} / RMSD {top['filter_rmsd']:.2f} A")

### 이 챕터에서 확인한 것 (본문 8.7)

- 실행 스텝은 문장이 아니라 `steps.yaml` 이 사실이고, `antibody-anything` 은 design_folding 을 쓰지 않아요.
- 재설계는 CDR 구간(48~60aa) 안에 갇혀 있고 그 안에 자유 Cys 가 없어요 — 프로토콜의 Cys 금지가 실제로 걸린 증거예요.
- VH·VL 두 사슬의 framework 말단과 보존 Cys 2개가 모두 살아 있어 정상 그래프팅이에요.
- num_designs 30 은 데모 규모예요. 실전 항체 캠페인은 **수천 개**까지 올려, 결합·구조·개발성을 동시에 만족하는
  희귀한 후보를 꼬리에서 건집니다(본문 8.7).

다음 Ch.09 는 같은 그래프팅을 **사슬 하나(VHH)** 로 할 때 무엇이 쉬워지고 무엇이 어려워지는지 봐요.